# 🎙 한국어 목소리 복제 (무료 Colab)
폰 브라우저에서 위에서부터 **셀을 하나씩 ▶ 눌러 실행**하세요. (처음 한 번 10분 정도, 이후엔 더 빠름)

1. 상단 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 저장
2. 아래 셀 1 → 2 → 3 → 4 순서로 실행
3. 4번에서 나오는 `https://....gradio.live` 링크를 눌러 쓰기

등록한 목소리와 결과는 내 **구글 드라이브 `voice_clone` 폴더**에 저장돼서 다음에도 남아 있습니다.
본인 또는 동의받은 사람의 목소리만 사용하세요.

## 1. GPU 확인 + 구글 드라이브 연결

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi","-L"],capture_output=True,text=True).stdout or "⚠ GPU 없음: 런타임 유형을 T4 GPU로 바꾸세요")
from google.colab import drive
drive.mount('/content/drive')

## 2. 설치 (약 5~8분)

In [ ]:
%%bash
set -e
BRANCH=ccr-b6b2108c-tujej3
apt-get -qq install -y sox libsox-dev > /dev/null
pip -q install uv
rm -rf /content/Voice && git clone -q --depth 1 -b $BRANCH https://github.com/delveintona/Voice /content/Voice
mkdir -p /content/engine/repo
[ -d /content/engine/repo/CosyVoice ] || git clone -q --depth 1 --recursive https://github.com/FunAudioLLM/CosyVoice /content/engine/repo/CosyVoice
[ -d /content/venv ] || uv venv --python 3.10 /content/venv
U="uv pip install --python /content/venv/bin/python"
$U wheel "setuptools<81"
$U torch==2.3.1 torchaudio==2.3.1
$U --no-build-isolation openai-whisper==20231117
grep -v -E "^(--|deepspeed|tensorrt|onnxruntime-gpu|torch|torchaudio)" /content/engine/repo/CosyVoice/requirements.txt > /tmp/req.txt
echo "onnxruntime-gpu==1.18.0" >> /tmp/req.txt
$U -r /tmp/req.txt
echo "설치 끝"

## 3. 음성 모델 내려받기 (약 4GB)

In [ ]:
import subprocess
code = '''
import os
d = "/content/engine/models/Fun-CosyVoice3-0.5B"
if os.path.exists(d + "/cosyvoice3.yaml"):
    print("이미 있음")
else:
    try:
        from huggingface_hub import snapshot_download
        snapshot_download("FunAudioLLM/Fun-CosyVoice3-0.5B-2512", local_dir=d, ignore_patterns=["llm.rl.pt"])
    except Exception as e:
        print("HF 실패, modelscope로 재시도:", e)
        from modelscope import snapshot_download
        snapshot_download("FunAudioLLM/Fun-CosyVoice3-0.5B-2512", local_dir=d)
    print("완료")
'''
_ = subprocess.run(["/content/venv/bin/python","-c",code],check=True)

## 4. 실행 → 나오는 gradio.live 링크 열기

In [ ]:
%%bash
# 서버 출력이 셀 아래에 바로 보이도록 bash 셀로 실행합니다 (-u: 버퍼 없이 즉시 출력)
export VOICE_DIR=/content/drive/MyDrive/voice_clone VOICE_ENGINE=/content/engine GRADIO_ANALYTICS_ENABLED=False PYTHONUNBUFFERED=1
mkdir -p $VOICE_DIR
cd /content/Voice/scripts
/content/venv/bin/python -u voice_app.py